# AETHER — XGBoost AQI Forecasting & Explainability Demo
### ET AI Hackathon 2026 | Problem Statement 5

This notebook demonstrates the machine learning pipeline behind the **AETHER Hyperlocal Forecasting Engine**. We walk through:
1. Loading historical timeseries readings and weather patterns from the database.
2. Engineering features (cyclical temporal encodings, lag features, rolling statistics).
3. Training an XGBoost Regressor for multi-horizon forecasting (t+24h, t+48h, t+72h).
4. Evaluating performance metrics (RMSE, MAE, R²) against a seasonal persistence baseline.
5. Explaining feature importances to show what factors contribute most to urban pollution trends.

### 1. Database Connections & Environment Setup

In [ ]:
import os
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Include parent directory in sys.path to access app modules
sys.path.append(os.path.abspath('..'))
from app.database import SessionLocal
from app.models import Reading, Weather, Station

db = SessionLocal()
print("Session established. Database status: Connected.")

### 2. Engineering Features
We extract lagged AQI observations, rolling stats, and cyclical wind vectors.

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['hour'] = df['measured_at'].dt.hour
    df['month'] = df['measured_at'].dt.month
    df['day_of_week'] = df['measured_at'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_rush_hour'] = (((df['hour'] >= 7) & (df['hour'] <= 10)) | ((df['hour'] >= 17) & (df['hour'] <= 20))).astype(int)
    df['is_winter'] = df['month'].isin([11, 12, 1, 2]).astype(int)
    
    # Cyclical wind encoding
    if 'wind_dir' in df.columns:
        df['wind_dir_sin'] = np.sin(np.radians(df['wind_dir'].fillna(0)))
        df['wind_dir_cos'] = np.cos(np.radians(df['wind_dir'].fillna(0)))
        
    # Lag features
    for lag in [1, 6, 12, 24, 48]:
        df[f'aqi_lag_{lag}h'] = df['aqi'].shift(lag)
        
    # Rolling statistics
    df['aqi_mean_6h'] = df['aqi'].rolling(6, min_periods=1).mean()
    df['aqi_mean_24h'] = df['aqi'].rolling(24, min_periods=1).mean()
    df['aqi_std_24h'] = df['aqi'].rolling(24, min_periods=1).std().fillna(0)
    
    return df.dropna()

### 3. Training & Validation Setups
We load a city's historical readings (e.g. Kolkata) and split 80% for training and 20% for verification.

In [ ]:
from app.services.forecaster import _get_station_history, _get_weather_history

city = "Kolkata"
df_aqi = _get_station_history(city, hours=30*24, db=db)
df_weather = _get_weather_history(city, hours=30*24, db=db)

if not df_weather.empty:
    df_weather = df_weather.rename(columns={'recorded_at': 'measured_at'})
    df = pd.merge_asof(
        df_aqi.sort_values('measured_at'),
        df_weather.sort_values('measured_at'),
        on='measured_at',
        direction='nearest'
    )
else:
    df = df_aqi

df_feat = engineer_features(df)
print(f"Feature matrix compiled: {df_feat.shape[0]} rows, {df_feat.shape[1]} columns.")

### 4. Training the XGBoost Model for 24-Hour Horizon

In [ ]:
# Set t+24h target
df_feat['aqi_target_24h'] = df_feat['aqi'].shift(-24)
feature_cols = [c for c in df_feat.columns if c not in ['measured_at', 'aqi', 'pm25', 'pm10', 'aqi_target_24h']]
df_clean = df_feat.dropna()

split_idx = int(len(df_clean) * 0.8)
X_train, y_train = df_clean[feature_cols].iloc[:split_idx], df_clean['aqi_target_24h'].iloc[:split_idx]
X_test, y_test = df_clean[feature_cols].iloc[split_idx:], df_clean['aqi_target_24h'].iloc[split_idx:]

model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.08,
    max_depth=5,
    random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

preds = model.predict(X_test)
rmse = math.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"XGBoost t+24h Results:\n  RMSE: {rmse:.2f}\n  MAE: {mae:.2f}\n  R² Score: {r2:.3f}")

### 5. Explainability: Visualizing Feature Importance

In [ ]:
# Plotting XGBoost built-in feature importances
fig, ax = plt.subplots(figsize=(10, 6))
xgb.plot_importance(model, ax=ax, max_num_features=10, height=0.6, color='orange')
plt.title("AETHER Forecast Engine — Feature Importance Rank")
plt.show()

In [ ]:
db.close()
print("Work complete. Database connection closed successfully.")